# 01 · Policy ingestion and the Q&A dataset
**Travel Shield Policy Explainer - capstone.** This notebook turns the 47-page policy PDF into searchable chunks and explores the trilingual input/output dataset built from it.

Steps: PDF → clean pages → section-aware chunks (page + heading) → dataset EDA.

In [2]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src import config
pd.set_option("display.max_colwidth", 120)

## 1. PDF → pages → section-aware chunks
Footers and page headers are removed; each chunk keeps the heading it falls under so answers can cite *[Section 14, p. 27]*.

In [3]:
from src.ingest import extract_pages, build_chunks, save_chunks
pages = extract_pages()
chunks = build_chunks(pages)
save_chunks(chunks)
print(len(pages), "pages ->", len(chunks), "chunks")
pd.DataFrame(chunks)[["chunk_id", "page", "section"]].sample(8, random_state=1)

47 pages -> 248 chunks


,chunk_id,page,section
67,p14-067,14,Section 5 - Compassionate Visit Benefit
247,p47-247,47,General Conditions - Civil Code 1250 Waiver Clause
210,p42-210,42,General Conditions - Proof of Loss
224,p44-224,44,General Conditions - Cancellation
90,p20-090,20,Section 7 - Personal Accident Benefit
222,p44-222,44,General Conditions - Cancellation
58,p13-058,13,Section 1 - Medical And Emergency Expenses Benefit
127,p27-127,27,Section 14 - Loss Or Damage Of Baggage And Personal Effects Benefit


In [4]:
ch = pd.DataFrame(chunks)
ch["chars"] = ch.text.str.len()
print(ch.chars.describe().round(0))
ch.section.str.replace(r" - .*", "", regex=True).value_counts().head(12)

count    248.0
mean     681.0
std      235.0
min      120.0
25%      475.0
50%      825.0
75%      866.0
max      899.0
Name: chars, dtype: float64


section
General Conditions                        48
Additional Exclusions                     46
Definitions                               35
Exclusions Applicable To All Sections     24
Section 7                                 10
Terms And Conditions Of Covid-19 Rider     7
Section 1                                  7
Section 14                                 7
Section 9                                  4
Section 10                                 4
Section 11                                 4
Section 18                                 4
Name: count, dtype: int64

## 2. The input/output dataset
Built by `scripts/build_dataset.py` from hand-written items in `data/qa/seed_part*.py`. Every item cites its section and page, and has the same question **and** answer in English, Tagalog and Taglish.

In [5]:
qa = pd.read_csv(config.QA_FILE)
clf = pd.read_csv(config.CLASSIFIER_FILE)
print(f"{qa.item_id.nunique()} items, {len(qa)} Q&A pairs, {len(clf)} classifier rows")
qa.pivot_table(index="topic", columns="language", values="qa_id", aggfunc="count").assign(total=lambda d: d.sum(axis=1))

95 items, 285 Q&A pairs, 521 classifier rows


language,english,tagalog,taglish,total
topic,,,,
baggage,7,7,7,21
claims,11,11,11,33
covid,7,7,7,21
dental,2,2,2,6
eligibility,7,7,7,21
evacuation,3,3,3,9
exclusions,8,8,8,24
flight_delay,5,5,5,15
hospital_family,4,4,4,12


In [6]:
qa[qa.item_id == "BAG-04"][["language", "input", "output"]]

,language,input,output
135,english,My luggage didn't arrive with my flight. What can I claim?,Baggage Delay pays for essential clothing and toiletries if your checked-in baggage is delayed more than 6 hours aft...
136,tagalog,Hindi dumating ang bagahe ko kasabay ng flight. Ano ang maaari kong i-claim?,Ang Baggage Delay ay nagbabayad ng mahahalagang damit at toiletries kung ang checked-in na bagahe ay naantala nang h...
137,taglish,Hindi dumating yung luggage ko pagdating ko. Ano pwede i-claim?,Ang Baggage Delay ay nagbabayad ng essential clothes at toiletries kung delayed ang checked-in baggage mo ng more th...


In [7]:
qa["q_words"] = qa.input.str.split().str.len()
qa["a_words"] = qa.output.str.split().str.len()
qa.groupby("language")[["q_words", "a_words"]].mean().round(1)

,q_words,a_words
language,,
english,9.7,49.7
tagalog,10.2,50.9
taglish,9.3,45.0


## 3. Split
Grouped by item (≈20% of each topic's items in test). No translation of a test question appears in training, so the evaluation measures generalisation to *new* questions.

In [8]:
qa.groupby(["split", "language"]).size().unstack()

language,english,tagalog,taglish
split,,,
test,19,19,19
train,76,76,76


## 4. Chat-format JSONL
`train.jsonl` / `test.jsonl` hold system/user/assistant records, usable as few-shot examples or for supervised fine-tuning.

In [ ]:
print(open(config.DATA_QA / "train.jsonl", encoding="utf-8").readline()[:600])

{"messages": [{"role": "system", "content": "You are the Travel Shield Policy Explainer for Prudential Guarantee's Travel Shield travel insurance. Answer only from the policy wording, cite the section and page, and reply in the same language the customer used: English, Tagalog, or Taglish."}, {"role": "user", "content": "What medical expenses are covered if I get sick or injured abroad?"}, {"role": "assistant", "content": "Section 1 pays, up to the limit in your Policy Schedule, the necessary and reasonable cost of medical, surgical and hospital treatment and ambulance costs if you become ill,


: 